# 03 — Utility

Comandi di servizio, uno per cella: controllare l'ambiente, seguire uno sweep,
guardare risultati e artefatti, fare pulizia. Nessuna cella misura niente, e
quelle che cancellano sono protette da un flag che va messo a `True` a mano.

I comandi girano sempre dalla root del repo, come se li scrivessi nel
terminale lì.

## Setup

`ROOT` è la root del repo. `sh()` lancia un comando di shell dalla root e ne
stampa l'output mentre arriva; `check=False` non ferma la cella se il comando
esce con errore (utile per i controlli che possono fallire). La cartella
`bin/` del venv viene messa nel PATH, come se il venv fosse attivato.

In [3]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "run.py").exists(), f"run.py non trovato in {ROOT}"
PY = sys.executable
# I binari del venv (benchmark_app di OpenVINO, wandb) nel PATH anche quando il
# kernel e' partito senza attivare il venv.
os.environ["PATH"] = f"{Path(PY).parent}:{os.environ.get('PATH', '')}"
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)


def sh(cmd: str, check: bool = True) -> int:
    """Esegue `cmd` in bash dalla root del repo, con output in streaming."""
    proc = subprocess.Popen(["bash", "-c", cmd], cwd=ROOT, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"uscito con codice {code}: {cmd}")
    return code


ROOT

PosixPath('/home/scolaroe/Desktop/yoloeval')

## 1. Ambiente

### Test

In [ ]:
sh(f"{PY} -m pytest -q tests")

### Versioni

Le librerie che contano per export e misura. TensorRT va letto due volte:
`trtexec` (di sistema, costruisce l'engine) e il pacchetto Python (carica
l'engine in validazione) devono avere **la stessa versione**, altrimenti
l'engine non si carica.

In [4]:
from importlib.metadata import PackageNotFoundError, version

for pkg in ("ultralytics", "torch", "onnx", "onnxruntime-gpu", "onnxruntime",
            "openvino", "nncf", "tensorrt-cu13", "tensorrt", "wandb"):
    try:
        print(f"{pkg:16s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:16s} -")

import torch
print(f"\ntorch CUDA       {torch.version.cuda}, GPU: "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'non disponibile'}")
sh("(command -v trtexec || echo /usr/src/tensorrt/bin/trtexec) | head -1 "
   "| xargs -I{} sh -c '{} --help 2>&1 | head -1'", check=False)

ultralytics      8.4.165
torch            2.12.1
onnx             1.23.0
onnxruntime-gpu  1.30.0
onnxruntime      -
openvino         2026.4.0
nncf             3.4.0
tensorrt-cu13    10.16.1.11
tensorrt         -
wandb            0.30.0

torch CUDA       13.0, GPU: NVIDIA RTX 6000 Ada Generation
&&&& RUNNING TensorRT.trtexec [TensorRT v101601] [b11] # /usr/bin/trtexec --help


0

### Tool di misura

I benchmark usano i binari dei vendor, non i binding Python: se uno manca,
tutte le celle di quel backend falliscono con `exit code 127`.

In [3]:
sh("""
for t in onnxruntime_perf_test trtexec benchmark_app pandoc; do
  p=$(command -v $t || true)
  [[ -z "$p" && $t == trtexec && -x /usr/src/tensorrt/bin/trtexec ]] && p=/usr/src/tensorrt/bin/trtexec
  printf '%-22s %s\n' "$t" "${p:-MANCANTE}"
done
""")

onnxruntime_perf_test  /home/scolaroe/Desktop/yoloeval/.venv/bin/onnxruntime_perf_test
trtexec                /usr/bin/trtexec
benchmark_app          /home/scolaroe/Desktop/yoloeval/.venv/bin/benchmark_app
pandoc                 /usr/bin/pandoc


0

### GPU: carico, temperatura, processi

In [13]:
sh("nvidia-smi", check=False)

Wed Sep 30 14:49:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 6000 Ada Gene...    On  |   00000000:6B:00.0  On |                  Off |
| 30%   44C    P8             29W /  300W |     892MiB /  49140MiB |     23%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

0

### Pacchetti TensorRT di sistema

Versione installata e stato di `hold`. Il repo usa TensorRT 10 (`x86_local.sh`):
se qui compare `11.x`, fp16 e int8 non si possono costruire con trtexec.

In [ ]:
sh("dpkg -l | awk '/libnvinfer|libnvonnxparsers/ {print $2, $3}'; "
   "echo '--- hold:'; apt-mark showhold | grep -E 'nvinfer|nvonnx' || echo nessuno",
   check=False)

### Provisioning della workstation

Crea o aggiorna il venv e installa TensorRT alla versione fissata. Chiede la
password di `sudo`, che un notebook non sa inserire: lancialo dal terminale.

In [ ]:
print("bash scripts/provision/x86_local.sh")

### Weights & Biases

Raggiungibilità del server e presenza delle credenziali. Il login va fatto dal
terminale: `.venv/bin/wandb login`.

In [3]:
sh("curl -sS -o /dev/null -w 'api.wandb.ai: HTTP %{http_code} in %{time_total}s\\n' "
   "--max-time 15 https://api.wandb.ai/healthz", check=False)
netrc = Path.home() / ".netrc"
logged = netrc.exists() and "api.wandb.ai" in netrc.read_text()
print("credenziali:", "presenti" if logged or os.environ.get("WANDB_API_KEY") else "assenti")

api.wandb.ai: HTTP 200 in 0.235916s
credenziali: assenti


## 2. Sweep in corso

### Processi attivi

Gli sweep lanciati con `run.py -m`, con la sottocartella di `multirun/` in cui
sta girando il job corrente (la sua cwd). Quella cartella non va cancellata
finché il processo è vivo.

In [ ]:
sh("""
pids=$(pgrep -f '^[^ ]*python[^ ]* run[.]py' || true)
[[ -z "$pids" ]] && { echo "nessuno sweep in esecuzione"; exit 0; }
for p in $pids; do
  echo "PID $p  avviato $(ps -o lstart= -p $p)"
  echo "  $(tr '\\0' ' ' < /proc/$p/cmdline)"
  echo "  cwd: $(readlink /proc/$p/cwd)"
done
""", check=False)

### Log dell'ultimo sweep

Le ultime righe di `run.log` di ogni job dell'ultima cartella in `multirun/`.
`N_RIGHE` controlla quante.

In [ ]:
N_RIGHE = 15

runs = sorted((ROOT / "multirun").glob("*/"), key=lambda p: p.name)
if not runs:
    print("nessuna cartella in multirun/")
else:
    last = runs[-1]
    print(f"sweep: {last.name}\n")
    for job in sorted((d for d in last.iterdir() if d.is_dir()), key=lambda p: int(p.name)):
        overrides = (job / ".hydra" / "overrides.yaml")
        print(f"=== job {job.name}: "
              f"{' '.join(l[2:] for l in overrides.read_text().splitlines()) if overrides.exists() else ''}")
        log = job / "run.log"
        if log.exists():
            print("".join(log.read_text(errors="replace").splitlines(keepends=True)[-N_RIGHE:]))

## 3. Risultati

Tutto quello che segue legge `results/*.json`, un file per cella. Una cella è
una combinazione completa di modello, quantizzazione, backend, board, profilo
di frequenza e compute target.

### Riepilogo per stato

In [ ]:
sys.path.insert(0, str(ROOT))
from tools.aggregate import load

df = pd.DataFrame(load(ROOT / "results"))
axes = ["model", "quantization", "backend", "board", "compute_target"]
print(df["status"].value_counts().to_string(), "\n")
df.pivot_table(index=["board", "backend"], columns="status", values="cell_id",
               aggfunc="count", fill_value=0)

### Celle fallite

Con il messaggio di errore accorciato alla prima riga. Per il testo completo
usa la cella successiva con il `cell_id`.

In [ ]:
failed = df[df["status"] == "failed"].copy()
failed["errore"] = failed["error"].astype(str).str.split("\n").str[0].str[:150]
failed[["cell_id", *axes, "errore"]].sort_values(axes)

### Celle saltate e motivo

In [ ]:
skipped = df[df["status"] == "skipped"]
skipped.groupby("reason").size().rename("celle").to_frame()

### Dettaglio di una cella

Il JSON completo di una cella e, se c'è, l'output grezzo del tool di misura
(`<cell_id>.raw.txt`).

In [ ]:
CELL_ID = ""   # es. "81a4410601ba"

path = ROOT / "results" / f"{CELL_ID}.json"
rec = json.loads(path.read_text())
rec.pop("config", None)   # la config completa e' lunga: toglierla per leggere il resto
print(json.dumps(rec, indent=2, ensure_ascii=False))
raw = path.with_suffix(".raw.txt")
if raw.exists():
    print("\n--- raw ---\n" + raw.read_text(errors="replace")[-3000:])

### Aggregazione

`results/*.json` → `data.parquet`, il file che leggono `01_analysis` e
`02_figures`. `--csv` scrive anche una copia CSV.

In [ ]:
sh(f"{PY} -m tools.aggregate --out data.parquet --csv")

### Report

Cartella datata in `reports/` con figure (PNG e PDF), tabelle CSV, markdown e
HTML; `reports/latest` punta all'ultima. Senza `pandoc` l'HTML viene saltato.

In [ ]:
sh(f"{PY} -m tools.report")

## 4. Artefatti

### Elenco

Pesi addestrati (mAP50 del training) ed export (esito della validazione e se la
testa è end-to-end). `val None e2e None` = export iniziato ma mai completato.

In [14]:
sh(f"{PY} -m tools.artifacts ls")

weights  yolo26n_aod4_e2_6e56352a  mAP50 0.6854  2026-09-28T11:16:58+02:00
weights  yolo26s_aod4_e2_cc470f8b  mAP50 0.7284  2026-09-28T11:19:26+02:00


0

### Dettaglio di un artefatto

Il `meta.json` completo: config, versioni, validazione, tempi. Lo slug è il
nome della cartella, come lo stampa `ls`.

In [ ]:
SLUG = ""   # es. "yolo26n_6e56352a_fp32_openvino_x86_64"
sh(f"{PY} -m tools.artifacts show {shlex.quote(SLUG)}")

### Export incompleti

Cartelle di `artifacts/exports/` senza `meta.json`: l'export si è fermato a
metà. Il benchmark le considera inesistenti.

In [ ]:
incomplete = [d.name for d in sorted((ROOT / "artifacts" / "exports").glob("*/"))
              if not (d / "meta.json").exists()]
print("\n".join(incomplete) or "nessuno")

### Errori di build TensorRT

Le righe `[E]` del `build.log` di trtexec per un export, più l'esito finale.
Senza `SLUG` le mostra per tutti gli export TensorRT che ne hanno.

In [ ]:
SLUG = ""   # es. "yolo26n_6e56352a_int8_tensorrt_x86_64"

logs = ([ROOT / "artifacts" / "exports" / SLUG / "build.log"] if SLUG
        else sorted((ROOT / "artifacts" / "exports").glob("*tensorrt*/build.log")))
for log in logs:
    lines = log.read_text(errors="replace").splitlines()
    errors = [l for l in lines if "[E]" in l or "&&&& FAILED" in l or "Assertion" in l]
    if errors or SLUG:
        print(f"=== {log.parent.name}")
        print("\n".join(errors[:20]) or "nessun errore")
        print()

### Prune: artefatti non usati da nessun risultato

Dry-run: elenca soltanto. Per cancellare davvero, `APPLICA = True`. Un export
TensorRT cancellato va ricompilato: leggi l'elenco prima.

In [ ]:
APPLICA = False
sh(f"{PY} -m tools.artifacts prune" + (" --apply" if APPLICA else ""))

## 5. Dataset

### Immagini per split

Conteggio di immagini ed etichette per ogni split dichiarato nel data yaml,
compreso `calib` (solo calibrazione INT8, disgiunto dagli altri).

In [ ]:
import yaml
from omegaconf import OmegaConf

ds_cfg = OmegaConf.load(ROOT / "conf" / "dataset" / "aod4.yaml")
data_root = ROOT / "data" / ds_cfg.name
spec = yaml.safe_load((data_root / "dataset.yaml").read_text())
IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
for split in ("train", "val", "test", "calib"):
    if split not in spec:
        continue
    img_dir = data_root / spec[split]
    lbl_dir = Path(str(img_dir).replace("/images/", "/labels/"))
    n_img = sum(1 for p in img_dir.rglob("*") if p.suffix.lower() in IMG)
    n_lbl = sum(1 for _ in lbl_dir.rglob("*.txt")) if lbl_dir.exists() else 0
    print(f"{split:6s} {n_img:6d} immagini  {n_lbl:6d} etichette  ({spec[split]})")
print("\nclassi:", spec.get("names"))

## 6. Pulizia

Prima mostra cosa c'è e quanto pesa, poi cancella solo con `CONFERMA = True`.
`.venv/` e `data/` non vengono mai toccati. Con uno sweep in esecuzione la
cella si rifiuta: gli toglierebbe la cartella da sotto.

### Spazio occupato

In [ ]:
sh("du -sh artifacts results reports multirun outputs data.parquet 2>/dev/null || true")

### Log di Hydra degli sweep terminati

Cancella le cartelle di `multirun/` e `outputs/` tranne quelle in uso da un
processo attivo.

In [ ]:
CONFERMA = False

active = set()
for pid in subprocess.run(["pgrep", "-f", "^[^ ]*python[^ ]* run[.]py"], capture_output=True, text=True).stdout.split():
    try:
        active.add(Path(f"/proc/{pid}/cwd").resolve())
    except OSError:
        pass
targets = [d for base in ("multirun", "outputs") for d in sorted((ROOT / base).glob("*/"))
           if not any(a == d or d in a.parents for a in active)]
print("\n".join(str(t.relative_to(ROOT)) for t in targets) or "niente da cancellare")
if CONFERMA and targets:
    import shutil
    for t in targets:
        shutil.rmtree(t)
    print(f"\ncancellate {len(targets)} cartelle")

### Pulizia completa

Artefatti (pesi ed export: poi si riaddestra tutto), risultati, report, log di
Hydra, `data.parquet`, pesi pretrained scaricati nella root. Rifiuta di partire
se c'è uno sweep attivo.

In [1]:
CONFERMA = False

cmd = ("rm -rf artifacts/* results/* reports/* multirun/ outputs/ "
       "data.parquet data.csv benchmark_report.csv yolo26*.pt")
print(("eseguo: " if CONFERMA else "[dry-run] ") + cmd)
if CONFERMA:
    if subprocess.run(["pgrep", "-f", "^[^ ]*python[^ ]* run[.]py"], capture_output=True).returncode == 0:
        raise RuntimeError("sweep in esecuzione: fermalo prima di pulire")
    sh(cmd)

[dry-run] rm -rf artifacts/* results/* reports/* multirun/ outputs/ data.parquet data.csv benchmark_report.csv yolo26*.pt


## 7. Probe

Misura su una board quale granularità di canali (N) le conviene, con reti
dummy. Il risultato va in `results/probe/` ed è il valore di N che usa
`strategy=pit_auto`; va lanciato prima di cercare con quella strategia. Serve
un probe per ogni board, precisione e backend che si vogliono usare con
`pit_auto`. Su CPU `pit_auto` legge il probe di `cpu_1`, quindi quello è
l'unico indispensabile.

`probe_edge` è il preset leggero per Pi 5 e CPU della Jetson (20-30 minuti
per compute target e precisione), `probe` quello completo per GPU e
workstation. Il probe supporta solo `onnxruntime` e `onnxruntime_py`; quelli
già fatti vengono saltati (`+force=true` per rifarli).

In [ ]:
DRY_RUN = False          # False per misurare davvero

# (board, preset, freq_target, compute_target); compute_target None = tutti quelli della board
PROBES = [
    # ("rpi5",         "probe_edge", "max",     "cpu_1"),
    # ("jetson_orin",  "probe_edge", "maxn",    "cpu_1"),
    # ("jetson_orin",  "probe",      "maxn",    "gpu"),
    ("wks4_rtx6000", "probe",      "default", "cpu_1"),
    ("wks4_rtx6000", "probe",      "default", "gpu"),
]
PROBE_QUANTS = ["fp32", "int8"]
PROBE_BACKEND = "onnxruntime"

for board, preset, freq, ct in PROBES:
    cmd = [PY, "run.py", "-m", f"stage={preset}", f"hardware={board}",
           f"freq_target={freq}", f"quantization={','.join(PROBE_QUANTS)}",
           f"backend={PROBE_BACKEND}"]
    if ct:
        cmd.append(f"compute_target={ct}")
    if DRY_RUN:
        cmd.append("++dry_run=true")
    print(f"\n===== {board} @ {freq}: {preset} {ct or 'tutti'} =====")
    sh(shlex.join(cmd))


===== wks4_rtx6000 @ default: probe cpu_1 =====
[2026-09-30 17:15:46,851][HYDRA] Launching 2 jobs locally
[2026-09-30 17:15:46,852][HYDRA] 	#0 : stage=probe hardware=wks4_rtx6000 freq_target=default quantization=fp32 backend=onnxruntime compute_target=cpu_1
[2026-09-30 17:15:48,452][INFO][src.stages.probe] probe wks4_rtx6000_cpu_1_fp32_onnxruntime_a1f4c2b2: 864 misure (3 forme x 96 canali x 3 passate), wks4_rtx6000 cpu_1 @ default
[2026-09-30 17:15:48,748][WARNING][src.stages.benchmark] scheduling FIFO non disponibile su localhost: la misura avra' piu' jitter (serve CAP_SYS_NICE)
[2026-09-30 17:16:08,348][INFO][src.stages.probe] probe wks4_rtx6000_cpu_1_fp32_onnxruntime_a1f4c2b2: 43/864 (4%), passata 1, ~6 min alla fine
[2026-09-30 17:16:30,096][INFO][src.stages.probe] probe wks4_rtx6000_cpu_1_fp32_onnxruntime_a1f4c2b2: 86/864 (9%), passata 1, ~6 min alla fine
[2026-09-30 17:16:55,205][INFO][src.stages.probe] probe wks4_rtx6000_cpu_1_fp32_onnxruntime_a1f4c2b2: 129/864 (14%), passata 1

### Probe fatti

Un probe per riga: `n_opt` è la N che `pit_auto` userà per quella board,
compute target, precisione e backend (fra probe uguali vale il più recente).
Il PNG accanto al summary mostra latenza e throughput in funzione di C.

In [19]:
rows = [json.loads(p.read_text())
        for p in sorted((ROOT / "results" / "probe").glob("*.summary.json"))]
cols = ["board", "freq_target", "compute_target", "quantization", "backend",
        "n_opt", "noise_rel", "created_at", "probe_id"]
pd.DataFrame(rows, columns=cols).sort_values(cols[:5]) if rows else print("nessun probe")

,board,freq_target,compute_target,quantization,backend,n_opt,noise_rel,created_at,probe_id
0,wks4_rtx6000,default,cpu_1,fp32,onnxruntime,16,0.213440,2026-09-30T15:28:41+02:00,wks4_rtx6000_cpu_1_fp32_onnxruntime_b732e234
1,wks4_rtx6000,default,cpu_1,int8,onnxruntime,4,0.076644,2026-09-30T15:38:16+02:00,wks4_rtx6000_cpu_1_int8_onnxruntime_c87569d9
2,wks4_rtx6000,default,gpu,fp32,onnxruntime,4,0.205469,2026-09-30T15:48:54+02:00,wks4_rtx6000_gpu_fp32_onnxruntime_d2bf8ffa
3,wks4_rtx6000,default,gpu,int8,onnxruntime,4,0.074039,2026-09-30T15:59:19+02:00,wks4_rtx6000_gpu_int8_onnxruntime_982155e1
